Step 1 — Load raw files and normalize column names

In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("../sintef_project/data")
KERNEL = DATA / "kernel"
EXTENDED = DATA / "extended"

# --- Market price (hourly)
price = pd.read_csv(KERNEL / "Historical_day_ahead_price_2015_2025.csv", parse_dates=["date"])
price = price.rename(columns={"date": "ts", price.columns[1]: "price"})
price["price"] = price["price"].astype("float32")

# --- Inflow (hourly, wide: date | reservoir_1 ... reservoir_17 | r_Vest_Vassdraget)
inflow = pd.read_csv(KERNEL / "Historical_inflow_1958_2025.csv", parse_dates=["date"])
inflow = inflow.rename(columns={"date": "ts"})
for c in inflow.columns[1:]:
    inflow[c] = inflow[c].astype("float32")

# --- Storage (daily, wide)
storage = pd.read_csv(KERNEL / "Historical_volume_2015_2024.csv", parse_dates=["date"])
storage = storage.rename(columns={"date": "ts"})

# --- Water value (daily, wide)
water_value = pd.read_csv(KERNEL / "Synthetic_water_value_2015_2024.csv", parse_dates=["date"])
water_value = water_value.rename(columns={"date": "ts"})

# --- Minimum volume constraint (hourly, wide)
min_vol = pd.read_csv(EXTENDED / "Constraint_min_volume.csv", parse_dates=["date"])
min_vol = min_vol.rename(columns={"date": "ts"})

# --- Minimum flow constraint (hourly, wide)
min_flow = pd.read_csv(EXTENDED / "Constraint_min_flow.csv", parse_dates=["date"])
min_flow = min_flow.rename(columns={"date": "ts"})

# --- Unit commitment (weekly cases, wide: Run No | starttime | result_committed_*_t*)
uc = pd.read_csv(KERNEL / "Unit_commitment_decisions.csv", parse_dates=["starttime"])

for name, df in [("price", price), ("inflow", inflow), ("storage", storage),
                 ("water_value", water_value), ("min_vol", min_vol),
                 ("min_flow", min_flow), ("uc", uc)]:
    print(f"{name:12s} shape={df.shape}  range={df['ts'].min() if 'ts' in df else df['starttime'].min()} → "
          f"{df['ts'].max() if 'ts' in df else df['starttime'].max()}")

def to_utc(s):
    return pd.to_datetime(s, utc=True)

price["ts"]        = to_utc(price["ts"])
inflow["ts"]       = to_utc(inflow["ts"])
storage["ts"]      = to_utc(storage["ts"])
water_value["ts"]  = to_utc(water_value["ts"])
min_vol["ts"]      = to_utc(min_vol["ts"])
min_flow["ts"]     = to_utc(min_flow["ts"])
uc["starttime"]    = to_utc(uc["starttime"])

price        shape=(103062, 2)  range=2015-01-01 00:00:00+00:00 → 2025-12-31 23:45:00+00:00
inflow       shape=(594224, 19)  range=1958-01-01 00:00:00+00:00 → 2025-10-15 07:00:00+00:00
storage      shape=(3654, 18)  range=2015-01-01 00:00:00+00:00 → 2025-01-01 00:00:00+00:00
water_value  shape=(3654, 18)  range=2015-01-01 00:00:00+00:00 → 2025-01-01 00:00:00+00:00
min_vol      shape=(96432, 4)  range=2015-01-01 00:00:00+00:00 → 2025-12-31 23:00:00+00:00
min_flow     shape=(96432, 7)  range=2015-01-01 00:00:00+00:00 → 2025-12-31 23:00:00+00:00
uc           shape=(2922, 2355)  range=2015-01-01 00:00:00 → 2022-12-31 00:00:00


Step 2 — Extract the list of units from UC columns

In [7]:
import re
uc_cols = [c for c in uc.columns if c.startswith("result_committed_")]
pattern = re.compile(r"result_committed_(?P<plant>.+?)_(?P<unit>G\d+)_t(?P<t>\d+)")

units = sorted({(m["plant"], m["unit"]) for c in uc_cols for m in [pattern.match(c)]})
print(f"Found {len(units)} generating units:")
for u in units:
    print(" ", u)
# Expected: 14 units across 9 plants (Hogga, Tokke, Lio, Byrte, Vinje, Haukeli, Songa, Kjela, Vesle Kjela)

Found 14 generating units:
  ('Byrte', 'G1')
  ('Haukeli', 'G1')
  ('Hogga', 'G1')
  ('Kjela', 'G1')
  ('Lio', 'G1')
  ('Songa', 'G1')
  ('Tokke', 'G1')
  ('Tokke', 'G2')
  ('Tokke', 'G3')
  ('Tokke', 'G4')
  ('Vesle Kjela', 'G1')
  ('Vinje', 'G1')
  ('Vinje', 'G2')
  ('Vinje', 'G3')


Step 3 — Parse the topology YAML

In [8]:
import yaml

with open(EXTENDED / "Tokke_Vinje_topology.yaml") as f:
    topo = yaml.safe_load(f)

# --- Inspect structure ONCE
print(topo.keys())
# Usually something like: dict_keys(['reservoirs', 'plants', 'generators', 'connections', ...])

# --- 3a. Reservoir metadata
# Adjust these keys to match your YAML
reservoir_info = {}   # name -> {"max_vol": float, ...}
for r in topo.get("reservoirs", []):
    name = r["name"]
    reservoir_info[name] = {
        "max_vol": float(r.get("max_vol", r.get("max_volume", np.nan))),
        "min_vol": float(r.get("min_vol", 0.0)),
    }

# --- 3b. Plant -> directly-connected reservoirs (its "input" reservoirs)
# This is the most important mapping. Look for a field like "connected_reservoirs",
# "input_reservoirs", or derive from "connections".
plant_direct_reservoirs = {}
for p in topo.get("plants", []):
    name = p["name"]
    direct = p.get("connected_reservoirs") or p.get("input_reservoirs") or []
    plant_direct_reservoirs[name] = list(direct)

# --- 3c. Build the DAG of reservoir -> downstream reservoir / plant
# `connections` typically lists (from, to) pairs where `to` can be a reservoir or plant.
edges_res_to_res = []   # (upstream_res, downstream_res)
edges_res_to_plant = [] # (res, plant)
for c in topo.get("connections", []):
    src, dst = c.get("from"), c.get("to")
    if src in reservoir_info and dst in reservoir_info:
        edges_res_to_res.append((src, dst))
    elif src in reservoir_info and dst in plant_direct_reservoirs:
        edges_res_to_plant.append((src, dst))

# --- 3d. Upstream closure per reservoir
from collections import defaultdict
upstream_of = defaultdict(set)
for up, down in edges_res_to_res:
    upstream_of[down].add(up)

def closure(res, seen=None):
    """All reservoirs that can contribute water to `res` (including itself)."""
    if seen is None: seen = set()
    if res in seen: return seen
    seen.add(res)
    for up in upstream_of.get(res, []):
        closure(up, seen)
    return seen

upstream_closure = {r: closure(r) for r in reservoir_info}

# --- 3e. Plant -> upstream reservoir closure
plant_upstream_reservoirs = {}
for plant, direct in plant_direct_reservoirs.items():
    ups = set()
    for r in direct:
        ups |= upstream_closure.get(r, {r})
    plant_upstream_reservoirs[plant] = ups

# --- 3f. Unit -> plant (from UC columns, already done above)

dict_keys(['model', 'connections'])


Step 4 — Generate the long-format index (case × unit × hour)

In [ ]:
# Explode UC to long
uc_long = uc.melt(
    id_vars=["Run No", "starttime"],
    value_vars=uc_cols,
    var_name="col",
    value_name="target",
)
uc_long[["plant", "unit", "t"]] = uc_long["col"].str.extract(
    r"result_committed_(?P<plant>.+?)_(?P<unit>G\d+)_t(?P<t>\d+)"
)
uc_long["t"] = uc_long["t"].astype("int16")
uc_long["target"] = uc_long["target"].astype("int8")
uc_long["unit_id"] = uc_long["plant"] + "_" + uc_long["unit"]

# Timestamp per hour
uc_long["ts"] = uc_long["starttime"] + pd.to_timedelta(uc_long["t"], unit="h")

# Cast categoricals
uc_long["unit_id"] = uc_long["unit_id"].astype("category")
uc_long["plant"] = uc_long["plant"].astype("category")

print("Long UC rows:", len(uc_long))   # ≈ 3647 * 14 * 168 ≈ 8.58M (train+val)
uc_long = uc_long.drop(columns=["col"])
uc_long = uc_long.sort_values(["Run No", "unit_id", "t"]).reset_index(drop=True)

uc_long["ts"] = uc["starttime"] # already UTC if uc["starttime"] is UTC
# or directly:
uc_long["ts"] = uc_long["starttime"] + pd.to_timedelta(uc_long["t"], unit="h")
# this preserves the UTC dtype

assert uc_long.groupby("Run No")["t"].nunique().eq(168).all()
assert uc_long["target"].isin([0, 1]).all()

Long UC rows: 6872544


Step 5 — Reindex daily data to hourly (forward-fill)

In [10]:
def daily_to_hourly(df, start, end):
    """Reindex a daily wide table to hourly using forward-fill within day."""
    idx = pd.date_range(start, end, freq="h", name="ts")
    return df.set_index("ts").reindex(idx, method="ffill").reset_index()

h_start, h_end = price["ts"].min(), price["ts"].max()

storage_h = daily_to_hourly(storage, h_start, h_end)
wv_h      = daily_to_hourly(water_value, h_start, h_end)

Step 6 — Build reservoir-level hourly panel (all features per reservoir)

In [12]:
# --- 6a. Melt each wide reservoir table to (ts, reservoir, value)
def wide_to_long(df, value_name):
    m = df.melt(id_vars="ts", var_name="reservoir", value_name=value_name)
    m["reservoir"] = m["reservoir"].astype("category")
    m[value_name] = m[value_name].astype("float32")
    return m

inflow_l  = wide_to_long(inflow, "inflow")
storage_l = wide_to_long(storage_h, "storage")
wv_l      = wide_to_long(wv_h, "water_value")
minvol_l  = wide_to_long(min_vol, "min_volume")

# --- 6b. Merge on (ts, reservoir)
res_panel = (
    inflow_l
    .merge(storage_l,  on=["ts", "reservoir"], how="left")
    .merge(wv_l,       on=["ts", "reservoir"], how="left")
    .merge(minvol_l,   on=["ts", "reservoir"], how="left")
)
res_panel["min_volume"] = res_panel["min_volume"].fillna(0.0)
res_panel["storage"]    = res_panel["storage"].astype("float32")
res_panel["water_value"] = res_panel["water_value"].astype("float32")

# --- 6c. Derived reservoir-level features
res_panel["fill_ratio"] = (
    res_panel["storage"]
    / res_panel.groupby("reservoir")["storage"].transform("max")
).astype("float32")

res_panel["volume_margin"] = (
    res_panel["storage"] - res_panel["min_volume"]
).astype("float32")

print("res_panel:", res_panel.shape, "unique reservoirs:", res_panel["reservoir"].nunique())

res_panel: (10696032, 8) unique reservoirs: 18


Step 7 — Attach static metadata to each unit

In [13]:
# From the YAML's generator definitions
unit_meta = {}   # "Tokke_G1" -> {"p_max": ..., "p_min": ..., "start_cost": ..., "efficiency": ..., "plant": "Tokke"}
for g in topo.get("generators", []):
    uid = f"{g['plant']}_{g['name']}"  # adjust keys
    unit_meta[uid] = {
        "p_max":      float(g.get("p_max",  np.nan)),
        "p_min":      float(g.get("p_min",  np.nan)),
        "start_cost": float(g.get("start_cost", 0.0)),
        "efficiency": float(g.get("efficiency", np.nan)),
    }
unit_meta_df = pd.DataFrame.from_dict(unit_meta, orient="index").rename_axis("unit_id").reset_index()

Step 8 — The core: join features onto the long UC table

In [14]:
# --- 8a. Map plant -> lists of reservoirs (use tuples so pandas can hash them)
plant_to_direct  = {p: tuple(sorted(v)) for p, v in plant_direct_reservoirs.items()}
plant_to_upstream = {p: tuple(sorted(v)) for p, v in plant_upstream_reservoirs.items()}

# --- 8b. Precompute per-reservoir pivot for fast aggregation
# Wide hourly tables keyed by reservoir (faster than groupby on long)
res_hourly = res_panel.pivot(index="ts", columns="reservoir",
                             values=["inflow", "storage", "water_value", "min_volume",
                                     "fill_ratio", "volume_margin"])
res_hourly.columns = [f"{a}__{b}" for a, b in res_hourly.columns]
res_hourly = res_hourly.sort_index()

def plant_direct_features(plant, ts_index):
    """DataFrame indexed by ts: direct_total_inflow, direct_mean_fill_ratio, etc."""
    cols = plant_to_direct.get(plant, ())
    out = pd.DataFrame(index=ts_index)
    if not cols:
        return out
    inflow_c  = [f"inflow__{r}"       for r in cols]
    fr_c      = [f"fill_ratio__{r}"   for r in cols]
    vol_c     = [f"storage__{r}"      for r in cols]
    wv_c      = [f"water_value__{r}"  for r in cols]
    mv_c      = [f"volume_margin__{r}" for r in cols]

    out["direct_total_inflow"]      = res_hourly[inflow_c].sum(axis=1)
    out["direct_total_volume"]      = res_hourly[vol_c].sum(axis=1)
    out["direct_mean_fill_ratio"]   = res_hourly[fr_c].mean(axis=1)
    out["direct_min_volume_margin"] = res_hourly[mv_c].min(axis=1)
    # volume-weighted water value
    num = (res_hourly[vol_c].values * res_hourly[wv_c].values).sum(axis=1)
    den = res_hourly[vol_c].values.sum(axis=1)
    out["direct_wavg_water_value"] = num / np.where(den == 0, np.nan, den)
    return out

def plant_upstream_features(plant, ts_index):
    cols = plant_to_upstream.get(plant, ())
    out = pd.DataFrame(index=ts_index)
    if not cols:
        return out
    inflow_c = [f"inflow__{r}"      for r in cols]
    vol_c    = [f"storage__{r}"     for r in cols]
    wv_c     = [f"water_value__{r}" for r in cols]

    out["upstream_total_inflow"] = res_hourly[inflow_c].sum(axis=1)
    out["upstream_total_volume"] = res_hourly[vol_c].sum(axis=1)
    num = (res_hourly[vol_c].values * res_hourly[wv_c].values).sum(axis=1)
    den = res_hourly[vol_c].values.sum(axis=1)
    out["upstream_wavg_water_value"] = num / np.where(den == 0, np.nan, den)
    return out

uc_long["ts"] = uc_long["ts"].astype("datetime64[ns]")

# Unique timestamps we need
ts_index = pd.DatetimeIndex(sorted(uc_long["ts"].unique()))
print("Unique timestamps:", len(ts_index))

# Precompute per-plant feature blocks once, then broadcast via merge
plant_blocks = {}
for plant in uc_long["plant"].cat.categories:
    d = plant_direct_features(plant, ts_index)
    u = plant_upstream_features(plant, ts_index)
    plant_blocks[plant] = pd.concat([d, u], axis=1).reset_index().rename(columns={"index": "ts"})

parts = []
for plant, block in plant_blocks.items():
    sub = uc_long[uc_long["plant"] == plant].merge(block, on="ts", how="left")
    parts.append(sub)
long_df = pd.concat(parts, ignore_index=True)

long_df = long_df.merge(price, on="ts", how="left")

long_df = long_df.merge(unit_meta_df, on="unit_id", how="left")

Unique timestamps: 70272


ValueError: You are trying to merge on datetime64[ns] and datetime64[ns, UTC] columns. If you wish to proceed you should use pd.concat

Step 10 — Final column list for v1

In [ ]:
feature_cols = [
    # time
    "t",                # hour within week
    "unit_id",          # categorical
    "plant",            # categorical
    # basic
    "price",
    # reservoir direct
    "direct_total_inflow", "direct_total_volume",
    "direct_mean_fill_ratio", "direct_min_volume_margin",
    "direct_wavg_water_value",
    # reservoir upstream
    "upstream_total_inflow", "upstream_total_volume",
    "upstream_wavg_water_value",
    # static
    "p_max", "p_min", "start_cost", "efficiency",
]

keep = ["Run No", "ts", "target"] + feature_cols
long_df = long_df[keep]

for c in long_df.select_dtypes("float64").columns:
    long_df[c] = long_df[c].astype("float32")
for c in long_df.select_dtypes("int64").columns:
    long_df[c] = pd.to_numeric(long_df[c], downcast="integer")

long_df.to_parquet("train_long_v1.parquet", index=False)
print(long_df.shape)   # ~8.6M rows
print(long_df.dtypes)

Step 11 — Quick sanity checks (do these before modeling)

In [ ]:
# 1) Every case has 14 units × 168 hours
assert (long_df.groupby("Run No").size() == 14 * 168).all()

# 2) No NaNs in the essential features
print(long_df[["price", "direct_total_inflow", "direct_mean_fill_ratio"]].isna().mean())

# 3) Target distribution
print(long_df.groupby("unit_id", observed=True)["target"].mean().sort_values())

# 4) Sanity of water-value-weighted averages (should be in a plausible EUR/MWh range)
print(long_df[["direct_wavg_water_value", "upstream_wavg_water_value"]].describe())